# Tutorial 02 — Collagen Fiber Dispersion

**Question:** How do mean orientation and angular dispersion jointly affect a distributed fiber response?

All values in this notebook are synthetic and dimensionless.

In [ ]:
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials
print(Path(biomechanics_tutorials.__file__).resolve())

## 1. Import the educational model

The fiber orientation is axial, so angles separated by $180^\circ$ are equivalent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from biomechanics_tutorials.collagen_dispersion import (
    DispersionParameters, FiberMaterialParameters,
    axial_von_mises_density, nominal_stress,
    numerical_orientation_tensor, order_parameter, orientation_grid,
    orientation_tensor,
)
from biomechanics_tutorials.plotting import apply_tutorial_style

apply_tutorial_style()

## 2. Axial von Mises distributions

In [ ]:
theta = orientation_grid(1001)
mean = np.deg2rad(20.0)
for beta, style in zip([0, 1, 4, 12], ["-", "--", "-.", ":"]):
    rho = axial_von_mises_density(theta, mean, beta)
    plt.plot(np.rad2deg(theta), rho, linestyle=style, linewidth=2.3,
             label=f"β={beta}, S={float(order_parameter(beta)):.3f}")
plt.xlabel("Axial orientation θ (deg)")
plt.ylabel("Probability density ρ (rad⁻¹)")
plt.title("Concentration controls angular spread")
plt.legend()
plt.show()

### Checkpoint

Explain why larger $\beta$ means lower dispersion. What happens when $\beta=0$?

## 3. Orientation tensor verification

In [ ]:
beta = 4.0
analytical = orientation_tensor(mean, beta)
numerical = numerical_orientation_tensor(mean, beta, points=257)
print("Analytical tensor:\n", analytical)
print("Numerical tensor:\n", numerical)
print("trace =", np.trace(numerical))
print("maximum component error =", np.max(np.abs(analytical - numerical)))

## 4. Distributed mechanical response

In [ ]:
stretches = np.linspace(1.0, 1.25, 121)
material = FiberMaterialParameters()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
for mean_angle, axis, title in [
    (0.0, axes[0], "Mean aligned with loading"),
    (np.deg2rad(45), axes[1], "Mean oblique to loading"),
]:
    for beta, style in zip([0, 1, 4, 12], ["-", "--", "-.", ":"]):
        distribution = DispersionParameters(
            mean_angle=mean_angle, concentration=beta, quadrature_points=1001
        )
        stress = nominal_stress(stretches, distribution, material)
        axis.plot(stretches, stress, linestyle=style, linewidth=2.2, label=f"β={beta}")
    axis.set_xlabel("Stretch λ")
    axis.set_ylabel("Nominal stress P")
    axis.set_title(title)
    axis.legend()
fig.tight_layout()
plt.show()

### Interpretation prompt

Why is the curve ordering different between the aligned and oblique cases? Avoid the general statement that dispersion always softens tissue.

## 5. Quadrature convergence

In [ ]:
reference = nominal_stress(
    stretches, DispersionParameters(mean_angle=np.deg2rad(37), concentration=7, quadrature_points=8193)
)
for points in [33, 65, 129, 257, 513, 1025]:
    candidate = nominal_stress(
        stretches, DispersionParameters(mean_angle=np.deg2rad(37), concentration=7, quadrature_points=points)
    )
    error = np.max(np.abs(candidate - reference))
    print(f"N={points:4d}  max error={error:.3e}")

## 6. Independent investigation

Choose one mean angle between $0^\circ$ and $90^\circ$. Determine whether increasing concentration raises or lowers the stress at $\lambda=1.18$. Then change the fiber stiffness and test whether your qualitative conclusion remains unchanged.

**Evidence boundary:** these are synthetic computational results, not tissue validation.